In [3]:
# ! pip install implicit
# ! pip install umap-learn


In [ ]:
'''
Data:
  История оценок фильмов пользователей

Input:
  История оценок конкретного пользователя

Output:
  Список наиболее подходящих ему фильмов
'''

In [7]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
from implicit.als import AlternatingLeastSquares
from IPython.display import display
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import umap

In [5]:
s = '''
Пользователь	Интерстеллар	Звездный войны	Дюна	Побег из Шоушенка	Зеленая миля	1+1
Алиса	3	5		1	1
Боб	4	5	5	2	1	2
Чарли	3	5	4	2		3
Мэллори	5	5	4	2		3
Чак	1	1		5	5
Дейв	3	3	2	3	5	5
Ева	1	1	2	5	4	4
Пэгги				5	5	5
'''

df = pd.read_csv(io.StringIO(s.strip()), sep='\t', index_col='Пользователь')
df

,Интерстеллар,Звездный войны,Дюна,Побег из Шоушенка,Зеленая миля,1+1
Пользователь,,,,,,
Алиса,3.0,5.0,NaN,1,1.0,NaN
Боб,4.0,5.0,5.0,2,1.0,2.0
Чарли,3.0,5.0,4.0,2,NaN,3.0
Мэллори,5.0,5.0,4.0,2,NaN,3.0
Чак,1.0,1.0,NaN,5,5.0,NaN
Дейв,3.0,3.0,2.0,3,5.0,5.0
Ева,1.0,1.0,2.0,5,4.0,4.0
Пэгги,NaN,NaN,NaN,5,5.0,5.0


In [12]:
df.shape

(8, 6)

In [20]:
'''
Разработаем подход из коллаборативной фильтрации
Для каждой пары (пользователь, фильм) мы будем предсказывать оценку.
1. Расcчитаем похожесть каждого пользователя с каждым по оценкам. Будем использовать косинусную близость.
2. Далее возьмем оценку этого фильма у всех пользователей и осредним с весами схожести пользователей
3. Мы получили предсказанные оценки всех фильмов этим пользователем
4. Теперь можно упорядочить все фильмы по оценке и взять топ N и таким образом сформировать выдачу
'''

'\nРазработаем подход из коллаборативной фильтрации\nНеобходимо для пары (пользователь, фильм) предсказать оценку\n1. Расcчитаем похожесть каждого пользователя с каждым по оценкам. Будем использовать косинусную близость.\n2. \n'

In [23]:
df_filled = df.fillna(0) # неизвестные оценки заполняем 0

In [24]:
df_filled.iloc[0]

,Алиса
Интерстеллар,3.0
Звездный войны,5.0
Дюна,0.0
Побег из Шоушенка,1.0
Зеленая миля,1.0
1+1,0.0


In [25]:
# вектор оценок пользователя Алиса
df_filled.iloc[0].values

array([3., 5., 0., 1., 1., 0.])

In [9]:
# Рассчитаем матрицу попарных расстояний между пользователями

user_similarity_df = pd.DataFrame(
     cosine_similarity(df_filled), index=df.index, columns=df.index
)
user_similarity_df

Пользователь,Алиса,Боб,Чарли,Мэллори,Чак,Дейв,Ева,Пэгги
Пользователь,,,,,,,,
Алиса,1.000000,0.769800,0.755929,0.787562,0.416025,0.592593,0.356966,0.192450
Боб,0.769800,1.000000,0.974707,0.974355,0.384308,0.744140,0.596462,0.333333
Чарли,0.755929,0.974707,1.000000,0.978059,0.314485,0.741930,0.603175,0.363696
Мэллори,0.787562,0.974355,0.978059,1.000000,0.312043,0.737558,0.566991,0.324785
Чак,0.416025,0.384308,0.314485,0.312043,1.000000,0.708784,0.821156,0.800641
Дейв,0.592593,0.744140,0.741930,0.737558,0.708784,1.000000,0.909914,0.833950
Ева,0.356966,0.596462,0.603175,0.566991,0.821156,0.909914,1.000000,0.945611
Пэгги,0.192450,0.333333,0.363696,0.324785,0.800641,0.833950,0.945611,1.000000


In [27]:
user_similarity_df.loc['Алиса'].sort_values(ascending=False)

,Алиса
Пользователь,
Алиса,1.000000
Мэллори,0.787562
Боб,0.769800
Чарли,0.755929
Дейв,0.592593
Чак,0.416025
Ева,0.356966
Пэгги,0.192450


In [29]:
df_filled.iloc[0].values

array([3., 5., 0., 1., 1., 0.])

In [31]:
df_filled.iloc[3].values

array([5., 5., 4., 2., 0., 3.])

In [32]:
df_filled.iloc[2].values

array([3., 5., 4., 2., 0., 3.])

In [33]:
user_similarity_df.loc['Чак'].sort_values(ascending=False)

,Чак
Пользователь,
Чак,1.000000
Ева,0.821156
Пэгги,0.800641
Дейв,0.708784
Алиса,0.416025
Боб,0.384308
Чарли,0.314485
Мэллори,0.312043


In [50]:
tmp = pd.DataFrame(user_similarity_df.loc['Алиса'].sort_values(ascending=False))
tmp.rename(columns={'Алиса': 'Схожесть'}, inplace=True)
tmp['Оценка'] = [df.loc[name, 'Дюна'] for name in tmp.index]
tmp.T

Пользователь,Алиса,Мэллори,Боб,Чарли,Дейв,Чак,Ева,Пэгги
Схожесть,1.0,0.787562,0.7698,0.755929,0.592593,0.416025,0.356966,0.19245
Оценка,NaN,4.000000,5.0000,4.000000,2.000000,NaN,2.000000,NaN


In [51]:
tmp.dropna().T

Пользователь,Мэллори,Боб,Чарли,Дейв,Ева
Схожесть,0.787562,0.7698,0.755929,0.592593,0.356966
Оценка,4.000000,5.0000,4.000000,2.000000,2.000000


In [52]:
weights = tmp.dropna()['Схожесть']
scores = tmp.dropna()['Оценка']

np.dot(scores, weights) / weights.sum()

np.float64(3.653886092519408)

In [53]:
'''
Функция, которая возвращает для пользователя топ фильмов
'''

def get_recs_coll_filtering(user, df):
    similarity_matrix = pd.DataFrame(
     cosine_similarity(df_filled), index=df.index, columns=df.index
    )
    # Оценки целевого пользователя
    user_ratings = df.loc[user]

    # Фильмы, которые пользователь еще НЕ оценивал
    unrated_movies = user_ratings[user_ratings.isna()].index

    recommendations = {}

    for movie in unrated_movies:
        # Оценки других пользователей для этого фильма
        movie_ratings = df[movie]

        # Пользователи, которые оценили этот фильм
        rated_users = movie_ratings.dropna().index

        if len(rated_users) > 0:
            # Веса (похожесть) только тех пользователей, кто оценил фильм
            weights = similarity_matrix.loc[user, rated_users]

            # Оценки этих пользователей
            scores = movie_ratings.loc[rated_users]

            # Прогнозируем оценку как взвешенное среднее
            predicted_score = np.dot(scores, weights) / weights.sum()
            recommendations[movie] = predicted_score

    # Сортируем фильмы по прогнозируемой оценке от лучшего к худшему
    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs

def get_rec_result(name):
    print(f'Рекомендации коллаборативной фильтрации для {name}')
    recs = get_recs_coll_filtering(name, df)
    for movie, score in recs:
        print(f"\t{movie}: прогнозируемая оценка = {score:.2f}")
    print('\n')

get_rec_result('Алиса')
get_rec_result('Чак')

Рекомендации коллаборативной фильтрации для Алиса
	Дюна: прогнозируемая оценка = 3.65
	1+1: прогнозируемая оценка = 3.33


Рекомендации коллаборативной фильтрации для Чак
	1+1: прогнозируемая оценка = 4.03
	Дюна: прогнозируемая оценка = 2.95




In [35]:
display(user_similarity_df.loc['Алиса'].sort_values(ascending=False))

,Алиса
Пользователь,
Алиса,1.000000
Мэллори,0.787562
Боб,0.769800
Чарли,0.755929
Дейв,0.592593
Чак,0.416025
Ева,0.356966
Пэгги,0.192450


In [54]:
display(user_similarity_df.loc['Чак'].sort_values(ascending=False))

,Чак
Пользователь,
Чак,1.000000
Ева,0.821156
Пэгги,0.800641
Дейв,0.708784
Алиса,0.416025
Боб,0.384308
Чарли,0.314485
Мэллори,0.312043


In [55]:
df

,Интерстеллар,Звездный войны,Дюна,Побег из Шоушенка,Зеленая миля,1+1
Пользователь,,,,,,
Алиса,3.0,5.0,NaN,1,1.0,NaN
Боб,4.0,5.0,5.0,2,1.0,2.0
Чарли,3.0,5.0,4.0,2,NaN,3.0
Мэллори,5.0,5.0,4.0,2,NaN,3.0
Чак,1.0,1.0,NaN,5,5.0,NaN
Дейв,3.0,3.0,2.0,3,5.0,5.0
Ева,1.0,1.0,2.0,5,4.0,4.0
Пэгги,NaN,NaN,NaN,5,5.0,5.0


In [121]:
def get_als_recommendations(user, df, topk=2):
    users = df.index.tolist()
    movies = df.columns

    user_id = users.index(user) # id текущего пользователя

    user_item_matrix = csr_matrix(df.fillna(0).values.astype(np.float32))

    model = AlternatingLeastSquares(
        factors=2,
        iterations=1, # на нашем небольшом примере достаточно одной итерации
        random_state=42,
    )
    model.fit(user_item_matrix)

    # Получаем рекомендации (массив_id_объектов, массив_скоров)
    ids, scores = model.recommend(
        user_id,
        user_item_matrix[user_id],
        N=topk,
        filter_already_liked_items=True,
    )

    print(f"Рекомендации ALS для {user}")
    for movie_id, score in zip(ids, scores):
        print(f"\t {movies[movie_id]} (Уверенность модели: {score:.4f})")
    print('\n\n')
    return model

model = get_als_recommendations("Алиса", df)
model = get_als_recommendations("Чак", df)

  0%|          | 0/1 [00:00<?, ?it/s]

Рекомендации ALS для Алиса
	 Дюна (Уверенность модели: 0.6761)
	 1+1 (Уверенность модели: 0.4983)





  0%|          | 0/1 [00:00<?, ?it/s]

Рекомендации ALS для Чак
	 1+1 (Уверенность модели: 0.7193)
	 Дюна (Уверенность модели: 0.5228)





In [89]:
def plot_emb(movie_vectors, movie_names):
    # 1. Автоматически делим фильмы на 2 кластера с помощью K-Means
    kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
    cluster_labels = kmeans.fit_predict(movie_vectors)

    df_plot = pd.DataFrame(
        {
            "X": movie_vectors[:, 0],
            "Y": movie_vectors[:, 1],
            "Название": movie_names,
            "Кластер": [
                f"Группа {c}" for c in cluster_labels
            ],  # преобразуем в текст для дискретной легенды
        }
    )

    fig = px.scatter(
        df_plot,
        x="X",
        y="Y",
        text="Название",
        color="Кластер",
        hover_name="Название",
        title="Интерактивная карта эмбеддингов",
        labels={"X": "Скрытый фактор 1", "Y": "Скрытый фактор 2"},
    )

    fig.update_traces(
        textposition="top center",
        marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")),  # Размер точек
    )

    fig.update_layout(
        xaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor="LightGray"),
        yaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor="LightGray"),
        plot_bgcolor="white",  # Белый фон
        width=900,
        height=600,
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="WhiteSmoke")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="WhiteSmoke")
    return fig

plot_emb(model.item_factors, df.columns.tolist())

In [126]:
import numpy as np
from implicit.evaluation import train_test_split, precision_at_k, ndcg_at_k
from implicit.als import AlternatingLeastSquares

# 1. Делим данные на обучающую и тестовую выборки (80% / 20%)
train_set, test_set = train_test_split(csr_matrix(df.fillna(0).values.astype(np.float32)), train_percentage=0.8)

# 2. Инициализируем и обучаем модель (например, ALS)
model = AlternatingLeastSquares(factors=64, iterations=15)
model.fit(train_set)

# 3. Вычисляем метрики на тестовой выборке для Top-10
# Обратите внимание: test_user_items и train_user_items должны иметь тип CSR (scipy)
K = 10
precision = precision_at_k(model, train_set, test_set, K=K)
ndcg = ndcg_at_k(model, train_set, test_set, K=K)

print(f"Precision@{K}: {precision:.4f}")
print(f"NDCG@{K}: {ndcg:.4f}")

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

Precision@10: 2.0769
NDCG@10: 1.4662


In [91]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

def convert_df(df_matrix):
    users = df_matrix.index.tolist()
    movies = df_matrix.columns.tolist()
    # Преобразуем в длинный формат (взаимодействия с оценкой > 0)
    interactions = []
    for u_idx, user in enumerate(users):
        for m_idx, movie in enumerate(movies):
            rating = df_matrix.loc[user, movie]
            if rating > 0:
                interactions.append((u_idx, m_idx, 1.0)) # 1.0 — факт позитивного взаимодействия

    df_train = pd.DataFrame(interactions, columns=["user_id", "item_id", "label"])
    return users, movies, df_train

def convert_ratings(df):
  movies = sorted(df['movieId'].unique().tolist())
  users = sorted(df['userId'].unique().tolist())

  movies_idx = {x: i for i, x in enumerate(movies)}
  users_idx = {x: i for i, x in enumerate(users)}

  lines = []
  for i, user, movie, rating in tqdm(df.itertuples()):
    lines.append((users_idx[user], movies_idx[movie], rating))
  return users, movies, pd.DataFrame(lines, columns=["user_id", "item_id", "label"])


# 2. Обертка данных в Dataset для PyTorch
class RecommendationDataset(Dataset):
    def __init__(self, dataframe):
        self.users = torch.tensor(dataframe["user_id"].values, dtype=torch.long)
        self.items = torch.tensor(dataframe["item_id"].values, dtype=torch.long)
        self.labels = torch.tensor(dataframe["label"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

# 3. Архитектура двухбашенной модели DSSM
class DSSMModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=8):
        super(DSSMModel, self).__init__()
        # Базовая башня Пользователя
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_nn = nn.Sequential(
            nn.Linear(embedding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8) # Финальный размер эмбеддинга (N-мерное пространство)
        )

        # Базовая башня Item
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.item_nn = nn.Sequential(
            nn.Linear(embedding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

    def forward(self, user_ids, item_ids):
        # Пропускаем пользователя через его башню
        u_emb = self.user_embedding(user_ids)
        u_vector = self.user_nn(u_emb)

        # Пропускаем товар через его башню
        i_emb = self.item_embedding(item_ids)
        i_vector = self.item_nn(i_emb)

        # Считаем косинусное сходство между векторами башен
        u_vector_norm = nn.functional.normalize(u_vector, p=2, dim=1)
        i_vector_norm = nn.functional.normalize(i_vector, p=2, dim=1)

        # Скалярное произведение строк (dot product)
        similarity = torch.sum(u_vector_norm * i_vector_norm, dim=1)
        return similarity

def fit_model(users, movies, dataloader, epochs, embedding_dim):
    # Инициализация
    model = DSSMModel(num_users=len(users), num_items=len(movies), embedding_dim=embedding_dim)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters())

    # Обучение модели
    model.train()
    for epoch in range(epochs):
        for batch_users, batch_items, batch_labels in dataloader:
            optimizer.zero_grad()
            predictions = model(batch_users, batch_items)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
    return model

def get_dssm_model_emb(df, need_convert_ratings=False, epochs=10, embedding_dim=8):
    if need_convert_ratings:
      users, movies, df_train = convert_ratings(df)
    else:
      users, movies, df_train = convert_df(df)

    dataset = RecommendationDataset(df_train)
    dataloader = DataLoader(dataset, batch_size=1)
    model = fit_model(users, movies, dataloader, epochs, embedding_dim)

    model.eval()
    with torch.no_grad():
        all_item_ids = torch.tensor(list(range(len(movies))), dtype=torch.long)
        item_embs = model.item_embedding(all_item_ids)
        movie_embeddings = model.item_nn(item_embs).numpy()

    print(f"Форма матрицы эмбеддингов: {movie_embeddings.shape}")
    return movie_embeddings, movies

plot_emb(get_dssm_model_emb(df, embedding_dim=8, epochs=10)[0], df.columns.tolist())

Форма матрицы эмбеддингов: (6, 8)


In [ ]:
import pandas as pd
a = pd.read_csv('https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/IMDB%20top%201000.csv')
a

,Unnamed: 0,Title,Certificate,Duration,Genre,Rate,Metascore,Description,Cast,Info
0,0,1. The Shawshank Redemption (1994),R,142 min,Drama,9.3,80.0,Two imprisoned men bond over a number of years...,"Director: Frank Darabont | Stars: Tim Robbins,...","Votes: 2,295,987 | Gross: $28.34M"
1,1,2. The Godfather (1972),R,175 min,"Crime, Drama",9.2,100.0,The aging patriarch of an organized crime dyna...,Director: Francis Ford Coppola | Stars: Marlon...,"Votes: 1,584,782 | Gross: $134.97M"
2,2,3. The Dark Knight (2008),PG-13,152 min,"Action, Crime, Drama",9.0,84.0,When the menace known as the Joker wreaks havo...,Director: Christopher Nolan | Stars: Christian...,"Votes: 2,260,649 | Gross: $534.86M"
3,3,4. The Godfather: Part II (1974),R,202 min,"Crime, Drama",9.0,90.0,The early life and career of Vito Corleone in ...,Director: Francis Ford Coppola | Stars: Al Pac...,"Votes: 1,107,253 | Gross: $57.30M"
4,4,5. The Lord of the Rings: The Return of the Ki...,PG-13,201 min,"Action, Adventure, Drama",8.9,94.0,Gandalf and Aragorn lead the World of Men agai...,"Director: Peter Jackson | Stars: Elijah Wood, ...","Votes: 1,614,369 | Gross: $377.85M"
...,...,...,...,...,...,...,...,...,...,...
995,995,398. Scent of a Woman (1992),R,156 min,Drama,8.0,NaN,A prep school student needing money agrees to ...,"Director: Martin Brest | Stars: Al Pacino, Chr...","Votes: 256,515 | Gross: $63.90M"
996,996,399. Aladdin (1992),G,90 min,"Animation, Adventure, Comedy",8.0,86.0,A kindhearted street urchin and a power-hungry...,"Directors: Ron Clements, John Musker | Stars: ...","Votes: 367,489 | Gross: $217.35M"
997,997,400. JFK (1991),R,189 min,"Drama, History, Thriller",8.0,72.0,New Orleans District Attorney Jim Garrison dis...,"Director: Oliver Stone | Stars: Kevin Costner,...","Votes: 139,634 | Gross: $70.41M"
998,998,301. Nights of Cabiria (1957),Not Rated,110 min,Drama,8.1,NaN,A waifish prostitute wanders the streets of Ro...,Director: Federico Fellini | Stars: Giulietta ...,"Votes: 42,160 | Gross: $0.75M"


In [ ]:
# https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/imdb_top_2025.csv
# https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/IMDB%20top%201000.csv
# https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/imdb_top_2025.csv

In [62]:
c = pd.read_csv('https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/imdb_top_2025.csv')
c

,Rank,Title,IMDb Rating,Votes,Poster URL,Video URL,Meta Score,Tags,Director,Description,Writers,Stars,Summary,Worldwide Gross
0,1,Once Upon a Time... in Hollywood,7.6,927K,https://www.imdb.com/title/tt7131622/mediaview...,https://imdb-video.media-imdb.com/vi1385741849...,84.0,"""Period Drama, Showbiz Drama, Comedy, Drama""",Quentin Tarantino,"""As Hollywood's Golden Age is winding down dur...",Quentin Tarantino,"""Leonardo DiCaprio, Brad Pitt, Margot Robbie""","""Reviewers say 'Once Upon a Time in Hollywood'...",-
1,2,Mission: Impossible - Dead Reckoning Part One,7.6,311K,https://www.imdb.com/title/tt9603212/mediaview...,https://imdb-video.media-imdb.com/vi3500918553...,81.0,"""Action Epic, Adventure Epic, Spy, Action, Adv...",Christopher McQuarrie,Ethan Hunt and his IMF team must track down a ...,"""Bruce Geller, Christopher McQuarrie, Erik Jen...","""Tom Cruise, Hayley Atwell, Ving Rhames""","""Reviewers say 'Mission: Impossible - Dead Rec...",-
2,3,John Wick: Chapter 4,7.6,392K,https://www.imdb.com/title/tt10366206/mediavie...,https://imdb-video.media-imdb.com/vi289916185/...,78.0,"""Action Epic, Gun Fu, One,Person Army Action, ...",Chad Stahelski,"""John Wick uncovers a path to defeating The Hi...","""Shay Hatten, Michael Finch, Derek Kolstad""","""Keanu Reeves, Laurence Fishburne, George Geor...","""Reviewers say 'John Wick: Chapter 4' is laude...",-
3,4,Watchmen,7.6,603K,https://www.imdb.com/title/tt0409459/mediaview...,https://imdb-video.media-imdb.com/vi240565017/...,56.0,"""Dystopian Sci,Fi, Superhero, Action, Drama, M...",Zack Snyder,"""In a version of 1985 where superheroes exist-...","""Dave Gibbons, David Hayter, Alex Tse""","""Jackie Earle Haley, Patrick Wilson, Carla Gug...","""Reviewers say 'Watchmen' is acclaimed for its...",-
4,5,The Fifth Element,7.6,533K,https://www.imdb.com/title/tt0119116/mediaview...,https://imdb-video.media-imdb.com/vi854720793/...,52.0,"""Sci,Fi Epic, Space Sci,Fi, Action, Adventure,...",Luc Besson,"""In the colorful future- a cab driver unwittin...","""Luc Besson, Robert Mark Kamen""","""Bruce Willis, Milla Jovovich, Gary Oldman""",-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,946,Grave of the Fireflies,8.5,361K,https://www.imdb.com/title/tt0095327/mediaview...,https://imdb-video.media-imdb.com/vi4000957977...,94.0,"Adult Animation, Anime, Hand,Drawn Animation, ...",Isao Takahata,A young boy and his little sister struggle to ...,"Akiyuki Nosaka, Isao Takahata","Tsutomu Tatsumi, Ayano Shiraishi, Akemi Yamaguchi",Reviewers say 'Hotaru no haka' is a poignant f...,NaN
946,947,American History X,8.5,1.2M,https://www.imdb.com/title/tt0120586/mediaview...,https://imdb-video.media-imdb.com/vi854001177/...,62.0,"Prison Drama, Psychological Drama, Tragedy, Cr...",Tony Kaye,A white supremacist is released from prison a ...,David McKenna,"Edward Norton, Edward Furlong, ""Beverly DAngelo""",Reviewers say 'American History X' is a powerf...,NaN
947,948,The Pianist,8.5,982K,https://www.imdb.com/title/tt0253474/mediaview...,https://imdb-video.media-imdb.com/vi988938521/...,85.0,"Docudrama, Epic, Period Drama, Tragedy, War Ep...",Roman Polanski,During WWII- acclaimed Polish musician Wladysl...,"Ronald Harwood, Wladyslaw Szpilman","Adrien Brody, Thomas Kretschmann, Frank Finlay",Reviewers say 'The Pianist' is a harrowing fil...,NaN
948,949,The Intouchables,8.5,993K,https://www.imdb.com/title/tt1675434/mediaview...,https://imdb-video.media-imdb.com/vi59285529/1...,57.0,"Buddy Comedy, Docudrama, Comedy, Drama",Olivier Nakache,After he becomes a quadriplegic from a paragli...,"Olivier Nakache, Éric Toledano, Philippe Pozzo...","François Cluzet, Omar Sy, Anne Le Ny",Reviewers say 'The Intouchables' is celebrated...,NaN


In [ ]:
c.shape

(950, 14)

In [ ]:
c['Title']

,Title
0,Once Upon a Time... in Hollywood
1,Mission: Impossible - Dead Reckoning Part One
2,John Wick: Chapter 4
3,Watchmen
4,The Fifth Element
...,...
945,Grave of the Fireflies
946,American History X
947,The Pianist
948,The Intouchables


In [94]:
movies = pd.read_csv('https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/new_movies.csv')
ratings = pd.read_feather('https://github.com/submaps/recsys_workshop/raw/refs/heads/main/new_ratings_1M.feather')

In [67]:
movies

,movieId,title,genres,year,Title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story
1,6,Heat (1995),Action|Crime|Thriller,1995,Heat
2,16,Casino (1995),Crime|Drama,1995,Casino
3,17,Sense and Sensibility (1995),Drama|Romance,1995,Sense and Sensibility
4,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,1995,The Usual Suspects
...,...,...,...,...,...
1034,207313,Knives Out (2019),Comedy|Crime|Drama|Mystery|Thriller,2019,Knives Out
1035,208006,Chhichhore (2019),Comedy|Romance,2019,Chhichhore
1036,208435,Super 30 (2019),Drama,2019,Super 30
1037,208477,Kaithi (2019),Action|Thriller,2019,Kaithi


In [64]:
ratings

,userId,movieId,rating
4520003,105872,91529,4.5
1434387,33696,110,3.5
2640951,61998,3897,3.0
1059346,25162,6,2.0
978048,23258,1256,5.0
...,...,...,...
6667306,156274,1721,4.5
2459871,57913,55820,4.5
128127,3180,8957,3.5
6080976,142567,60069,4.0


In [68]:
ratings.shape

(1000000, 3)

In [71]:
ratings['userId'].nunique()

140270

In [72]:
ratings['movieId'].nunique()

943

In [75]:
movies[:20]

[1,
 6,
 16,
 17,
 50,
 58,
 105,
 110,
 111,
 123,
 150,
 199,
 215,
 223,
 235,
 261,
 277,
 290,
 293,
 306]

In [76]:
ratings

,userId,movieId,rating
4520003,105872,91529,4.5
1434387,33696,110,3.5
2640951,61998,3897,3.0
1059346,25162,6,2.0
978048,23258,1256,5.0
...,...,...,...
6667306,156274,1721,4.5
2459871,57913,55820,4.5
128127,3180,8957,3.5
6080976,142567,60069,4.0


In [116]:
%%time
def get_dssm_ratings_emb(n=1000):
  emb, movieIds = get_dssm_model_emb(ratings.head(n), need_convert_ratings=True, embedding_dim=8, epochs=10)
  movie_names_dict = dict(movies[['movieId', 'title']].values)
  emb_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42).fit_transform(emb)
  movieIds_names = [movie_names_dict.get(mid, 'Unknown') for mid in movieIds]
  return emb_2d, movieIds_names

emb_2d, movieIds_names = get_dssm_ratings_emb(n=100)
plot_emb(emb_2d, movieIds_names)

100it [00:00, 269730.16it/s]


Форма матрицы эмбеддингов: (88, 8)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



CPU times: user 1.95 s, sys: 3 ms, total: 1.95 s
Wall time: 1.96 s


In [ ]:
'https://raw.githubusercontent.com/submaps/recsys_workshop/refs/heads/main/imdb_top_2025.csv'

In [88]:
# ! pip install umap-learn

In [118]:
ratings

,userId,movieId,rating
4520003,105872,91529,4.5
1434387,33696,110,3.5
2640951,61998,3897,3.0
1059346,25162,6,2.0
978048,23258,1256,5.0
...,...,...,...
6667306,156274,1721,4.5
2459871,57913,55820,4.5
128127,3180,8957,3.5
6080976,142567,60069,4.0


In [120]:
ratings.reset_index(drop=True)

,userId,movieId,rating
0,105872,91529,4.5
1,33696,110,3.5
2,61998,3897,3.0
3,25162,6,2.0
4,23258,1256,5.0
...,...,...,...
999995,156274,1721,4.5
999996,57913,55820,4.5
999997,3180,8957,3.5
999998,142567,60069,4.0
